<a href="https://colab.research.google.com/github/Rini43/Asteroid_Diameter_Prediction/blob/main/Exit_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## Asteroides Diameter Predictor

# Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Read the data

In [ ]:
# Load the dataset
filepath = '/content/drive/MyDrive/Data/dataset.csv'
df_asteroid = pd.read_csv(filepath)
df_asteroid.head(20)

## EDA

In [ ]:
# Identifying numerical and categorical data
df_asteroid.info()

In [ ]:
# Identifying dataset size
df_asteroid.shape

In [ ]:
# Statistical Summary
df_asteroid.describe()

## Target Variable Analysis: Is Diameter Normally Distributed?

In [ ]:
# Prepare diameter data (remove missing values)
df_diameter = df_asteroid[df_asteroid['diameter'].notna()].copy()
diameter = df_diameter['diameter'].values

print("="*70)
print("NORMALITY TEST FOR TARGET VARIABLE (DIAMETER)")
print("="*70)
print(f"\nDataset Information:")
print(f"  Total records with diameter data: {len(diameter):,}")
print(f"  Mean: {diameter.mean():.2f} km")
print(f"  Median: {np.median(diameter):.2f} km")
print(f"  Std Dev: {diameter.std():.2f} km")
print(f"  Range: {diameter.min():.3f} - {diameter.max():.2f} km")

In [ ]:
# Statistical Tests for Normality
print("\n" + "="*70)
print("STATISTICAL NORMALITY TESTS")
print("="*70)

# 1. Shapiro-Wilk Test (works best for n < 5000)
# Sample data if too large
sample_size = min(5000, len(diameter))
diameter_sample = np.random.choice(diameter, size=sample_size, replace=False)

shapiro_stat, shapiro_p = stats.shapiro(diameter_sample)
print(f"\n1. Shapiro-Wilk Test (sample of {sample_size:,}):")
print(f"   Test Statistic: {shapiro_stat:.6f}")
print(f"   P-value: {shapiro_p:.2e}")
if shapiro_p < 0.05:
    print(f"   Conclusion: REJECT null hypothesis (p < 0.05)")
    print(f"   ➜ Data is NOT normally distributed")
else:
    print(f"   Conclusion: FAIL TO REJECT null hypothesis (p ≥ 0.05)")
    print(f"   ➜ Data is normally distributed")

# 2. Kolmogorov-Smirnov Test
ks_stat, ks_p = stats.kstest(diameter, 'norm', args=(diameter.mean(), diameter.std()))
print(f"\n2. Kolmogorov-Smirnov Test:")
print(f"   Test Statistic: {ks_stat:.6f}")
print(f"   P-value: {ks_p:.2e}")
if ks_p < 0.05:
    print(f"   Conclusion: REJECT null hypothesis (p < 0.05)")
    print(f"   ➜ Data is NOT normally distributed")
else:
    print(f"   Conclusion: FAIL TO REJECT null hypothesis (p ≥ 0.05)")
    print(f"   ➜ Data is normally distributed")

# 3. Anderson-Darling Test
anderson_result = stats.anderson(diameter)
print(f"\n3. Anderson-Darling Test:")
print(f"   Test Statistic: {anderson_result.statistic:.6f}")
print(f"   Critical Values: {anderson_result.critical_values}")
print(f"   Significance Levels: {anderson_result.significance_level}%")
if anderson_result.statistic > anderson_result.critical_values[2]:  # 5% level
    print(f"   Conclusion: REJECT null hypothesis")
    print(f"   ➜ Data is NOT normally distributed")
else:
    print(f"   Conclusion: FAIL TO REJECT null hypothesis")
    print(f"   ➜ Data is normally distributed")

# 4. Jarque-Bera Test
jb_stat, jb_p = stats.jarque_bera(diameter)
print(f"\n4. Jarque-Bera Test:")
print(f"   Test Statistic: {jb_stat:.6f}")
print(f"   P-value: {jb_p:.2e}")
if jb_p < 0.05:
    print(f"   Conclusion: REJECT null hypothesis (p < 0.05)")
    print(f"   ➜ Data is NOT normally distributed")
else:
    print(f"   Conclusion: FAIL TO REJECT null hypothesis (p ≥ 0.05)")
    print(f"   ➜ Data is normally distributed")

In [ ]:
# Skewness and Kurtosis Analysis
print("\n" + "="*70)
print("SKEWNESS AND KURTOSIS ANALYSIS")
print("="*70)

skewness = stats.skew(diameter)
kurtosis = stats.kurtosis(diameter)

print(f"\nSkewness: {skewness:.4f}")
print(f"  Interpretation:")
if abs(skewness) < 0.5:
    print(f"    • |Skewness| < 0.5: Fairly symmetrical")
elif abs(skewness) < 1:
    print(f"    • 0.5 < |Skewness| < 1: Moderately skewed")
else:
    print(f"    • |Skewness| > 1: Highly skewed")

if skewness > 0:
    print(f"    • Positive skew: Right tail is longer (tail extends to the right)")
    print(f"    • Most values concentrate on the LEFT, few large values on RIGHT")
else:
    print(f"    • Negative skew: Left tail is longer (tail extends to the left)")
    print(f"    • Most values concentrate on the RIGHT, few small values on LEFT")

print(f"\nKurtosis (Excess): {kurtosis:.4f}")
print(f"  Interpretation:")
if kurtosis > 0:
    print(f"    • Positive kurtosis: Heavy tails (Leptokurtic)")
    print(f"    • More outliers than normal distribution")
elif kurtosis < 0:
    print(f"    • Negative kurtosis: Light tails (Platykurtic)")
    print(f"    • Fewer outliers than normal distribution")
else:
    print(f"    • Zero kurtosis: Similar to normal distribution (Mesokurtic)")

print(f"\nNormal Distribution Benchmarks:")
print(f"  • Skewness ≈ 0 (ideally between -0.5 and +0.5)")
print(f"  • Kurtosis ≈ 0 (excess kurtosis should be close to 0)")

In [ ]:
# Visual Evidence 1: Histogram with Normal Distribution Overlay
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Normality Assessment: Target Variable (Diameter)', fontsize=16, fontweight='bold', y=0.995)

# Plot 1: Histogram with Normal Curve Overlay
ax1 = axes[0, 0]
n, bins, patches = ax1.hist(diameter, bins=100, density=True, alpha=0.7, color='skyblue', edgecolor='black', label='Data')

# Fit and plot normal distribution
mu, sigma = diameter.mean(), diameter.std()
x = np.linspace(diameter.min(), diameter.max(), 100)
ax1.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2.5, label='Normal Distribution')

ax1.set_xlabel('Diameter (km)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Density', fontsize=11, fontweight='bold')
ax1.set_title('Histogram vs Normal Distribution Curve', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.text(0.98, 0.97, 'Visual Interpretation:\n• Red curve = Expected normal distribution\n• Blue bars = Actual data\n• MISMATCH indicates non-normality',
        transform=ax1.transAxes, fontsize=9, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Plot 2: Log-scale Histogram
ax2 = axes[0, 1]
ax2.hist(diameter, bins=100, color='lightcoral', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Diameter (km)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frequency (log scale)', fontsize=11, fontweight='bold')
ax2.set_yscale('log')
ax2.set_title('Histogram with Log Scale (Reveals Right Skew)', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.text(0.98, 0.97, 'Right-skewed tail visible:\n• Heavy concentration on left\n• Long tail extending right',
        transform=ax2.transAxes, fontsize=9, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Plot 3: Q-Q Plot (Most important for normality)
ax3 = axes[1, 0]
stats.probplot(diameter, dist="norm", plot=ax3)
ax3.set_title('Q-Q Plot: Data vs Normal Distribution', fontsize=12, fontweight='bold')
ax3.set_xlabel('Theoretical Quantiles', fontsize=11, fontweight='bold')
ax3.set_ylabel('Sample Quantiles', fontsize=11, fontweight='bold')
ax3.grid(alpha=0.3)
ax3.text(0.05, 0.95, 'Interpretation:\n• Points on red line = Normal distribution\n• Deviation from line = Non-normality\n• Curved pattern = Skewness/Heavy tails',
        transform=ax3.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Plot 4: Density Plot with Normal Curve
ax4 = axes[1, 1]
diameter_series = pd.Series(diameter)
diameter_series.plot(kind='density', ax=ax4, color='blue', linewidth=2.5, label='Data Density')

# Plot normal distribution
x_normal = np.linspace(diameter.min(), diameter.max(), 100)
ax4.plot(x_normal, stats.norm.pdf(x_normal, mu, sigma), 'r--', linewidth=2.5, label='Normal Distribution')
ax4.fill_between(x_normal, stats.norm.pdf(x_normal, mu, sigma), alpha=0.2, color='red')

ax4.set_xlabel('Diameter (km)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Density', fontsize=11, fontweight='bold')
ax4.set_title('Kernel Density Estimate vs Normal Distribution', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(alpha=0.3)
ax4.text(0.98, 0.97, 'Right skew evident:\n• Peak shifts left\n• Long right tail',
        transform=ax4.transAxes, fontsize=9, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("VISUAL EVIDENCE INTERPRETATION")
print("="*70)
print("\n1. HISTOGRAM vs NORMAL CURVE:")
print("   • The red curve shows expected normal distribution")
print("   • The blue histogram shows actual data")
print("   • OBSERVATION: Clear mismatch - data doesn't follow bell curve")
print("   • The distribution has a heavy left concentration and long right tail")

print("\n2. LOG-SCALE HISTOGRAM:")
print("   • Reveals the degree of right skewness")
print("   • Shows there are many small asteroids and few large ones")

print("\n3. Q-Q PLOT (MOST IMPORTANT):")
print("   • If data were normal, points would follow the red line closely")
print("   • OBSERVATION: Points deviate significantly from the line")
print("   • Deviation pattern confirms non-normality and right skewness")
print("   • Curve upward at ends indicates heavy right tail")

print("\n4. DENSITY PLOT:")
print("   • Blue curve = actual data density")
print("   • Red dashed curve = expected normal distribution")
print("   • OBSERVATION: Significant differences in shape and location")

In [ ]:
# Visual Evidence 2: Box Plot and Violin Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distributional Characteristics: Evidence of Non-Normality', fontsize=14, fontweight='bold')

# Plot 1: Box Plot
bp = axes[0].boxplot(diameter, vert=True, patch_artist=True,
                      boxprops=dict(facecolor='lightblue', alpha=0.7),
                      medianprops=dict(color='red', linewidth=2.5),
                      flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5))
axes[0].set_ylabel('Diameter (km)', fontsize=11, fontweight='bold')
axes[0].set_title('Box Plot', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add annotations
q1 = np.percentile(diameter, 25)
q2 = np.percentile(diameter, 50)
q3 = np.percentile(diameter, 75)
iqr = q3 - q1

axes[0].text(1.15, q1, f'Q1: {q1:.2f}', fontsize=9, va='center')
axes[0].text(1.15, q2, f'Median: {q2:.2f}', fontsize=9, va='center', fontweight='bold', color='red')
axes[0].text(1.15, q3, f'Q3: {q3:.2f}', fontsize=9, va='center')

axes[0].text(0.02, 0.98, f'Outliers visible (red dots)\nIndicates non-normal tail behavior',
            transform=axes[0].transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Plot 2: Violin Plot
parts = axes[1].violinplot(diameter, vert=True, widths=0.7, showmeans=True, showmedians=True)
axes[1].set_ylabel('Diameter (km)', fontsize=11, fontweight='bold')
axes[1].set_title('Violin Plot (Shape Comparison)', fontsize=12, fontweight='bold')
axes[1].set_xticks([])
axes[1].grid(axis='y', alpha=0.3)

axes[1].text(0.02, 0.98, f'Asymmetric shape\nFat lower portion\nLong upper tail',
            transform=axes[1].transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Plot 3: Cumulative Distribution
sorted_diameter = np.sort(diameter)
cumulative = np.arange(1, len(sorted_diameter) + 1) / len(sorted_diameter)

axes[2].plot(sorted_diameter, cumulative, 'b-', linewidth=2, label='Actual CDF')

# Plot theoretical normal CDF
theoretical_cdf = stats.norm.cdf(sorted_diameter, mu, sigma)
axes[2].plot(sorted_diameter, theoretical_cdf, 'r--', linewidth=2, label='Normal CDF')

axes[2].set_xlabel('Diameter (km)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Cumulative Probability', fontsize=11, fontweight='bold')
axes[2].set_title('Cumulative Distribution Function', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)

axes[2].text(0.02, 0.98, f'CDF divergence\nfrom normal curve\nindicates non-normality',
            transform=axes[2].transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Visual Evidence 3: Comparison with Transformed Data
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Original vs Log-Transformed Data: Effects of Transformation', fontsize=14, fontweight='bold')

# Original Data
ax = axes[0, 0]
n, bins, patches = ax.hist(diameter, bins=100, density=True, alpha=0.7, color='skyblue', edgecolor='black')
x = np.linspace(diameter.min(), diameter.max(), 100)
ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2)
ax.set_title('Original: Histogram + Normal Curve', fontsize=11, fontweight='bold')
ax.set_xlabel('Diameter (km)')
ax.set_ylabel('Density')
ax.grid(alpha=0.3)

ax = axes[0, 1]
stats.probplot(diameter, dist="norm", plot=ax)
ax.set_title('Original: Q-Q Plot', fontsize=11, fontweight='bold')
ax.grid(alpha=0.3)

ax = axes[0, 2]
sorted_d = np.sort(diameter)
cum = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
ax.plot(sorted_d, cum, 'b-', linewidth=2, label='Actual')
ax.plot(sorted_d, stats.norm.cdf(sorted_d, mu, sigma), 'r--', linewidth=2, label='Normal')
ax.set_title('Original: CDF', fontsize=11, fontweight='bold')
ax.set_xlabel('Diameter (km)')
ax.set_ylabel('Cumulative Probability')
ax.legend()
ax.grid(alpha=0.3)

# Log-Transformed Data
log_diameter = np.log(diameter)
mu_log, sigma_log = log_diameter.mean(), log_diameter.std()

ax = axes[1, 0]
n, bins, patches = ax.hist(log_diameter, bins=100, density=True, alpha=0.7, color='lightgreen', edgecolor='black')
x_log = np.linspace(log_diameter.min(), log_diameter.max(), 100)
ax.plot(x_log, stats.norm.pdf(x_log, mu_log, sigma_log), 'r-', linewidth=2)
ax.set_title('Log-Transformed: Histogram + Normal Curve', fontsize=11, fontweight='bold')
ax.set_xlabel('Log(Diameter)')
ax.set_ylabel('Density')
ax.grid(alpha=0.3)

ax = axes[1, 1]
stats.probplot(log_diameter, dist="norm", plot=ax)
ax.set_title('Log-Transformed: Q-Q Plot', fontsize=11, fontweight='bold')
ax.grid(alpha=0.3)

ax = axes[1, 2]
sorted_log = np.sort(log_diameter)
cum_log = np.arange(1, len(sorted_log) + 1) / len(sorted_log)
ax.plot(sorted_log, cum_log, 'b-', linewidth=2, label='Actual')
ax.plot(sorted_log, stats.norm.cdf(sorted_log, mu_log, sigma_log), 'r--', linewidth=2, label='Normal')
ax.set_title('Log-Transformed: CDF', fontsize=11, fontweight='bold')
ax.set_xlabel('Log(Diameter)')
ax.set_ylabel('Cumulative Probability')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate skewness for both
print(f"\nTransformation Effect on Normality:")
print(f"\nOriginal Data:")
print(f"  Skewness: {stats.skew(diameter):.4f} (Highly right-skewed)")
print(f"  Kurtosis: {stats.kurtosis(diameter):.4f} (Heavy tails)")

print(f"\nLog-Transformed Data:")
print(f"  Skewness: {stats.skew(log_diameter):.4f} (Much closer to 0)")
print(f"  Kurtosis: {stats.kurtosis(log_diameter):.4f} (Closer to normal)")

print(f"\nObservation: Log transformation makes the distribution MUCH MORE NORMAL")

In [ ]:
# Final Summary
print("\n" + "="*70)
print("COMPREHENSIVE NORMALITY ASSESSMENT SUMMARY")
print("="*70)

print("\n📊 CONCLUSION: IS DIAMETER NORMALLY DISTRIBUTED?")
print("\n   ❌ NO - The target variable (asteroid diameter) is NOT normally distributed")

print("\n" + "-"*70)
print("STATISTICAL EVIDENCE:")
print("-"*70)
print(f"\n1. Normality Tests Results:")
print(f"   • Shapiro-Wilk p-value: < 0.05 ➜ REJECT normality")
print(f"   • Kolmogorov-Smirnov p-value: < 0.05 ➜ REJECT normality")
print(f"   • Jarque-Bera p-value: < 0.05 ➜ REJECT normality")
print(f"   • Anderson-Darling: Test statistic exceeds critical values ➜ REJECT normality")

print(f"\n2. Distributional Characteristics:")
print(f"   • Skewness: {stats.skew(diameter):.4f}")
print(f"     → Highly positive (RIGHT-SKEWED)")
print(f"     → Most asteroids are small, few are very large")
print(f"   • Kurtosis: {stats.kurtosis(diameter):.4f}")
print(f"     → High positive value (HEAVY TAILS)")
print(f"     → More extreme outliers than normal distribution")
print(f"   • Mean ({diameter.mean():.2f}) ≠ Median ({np.median(diameter):.2f})")
print(f"     → In normal distribution, mean ≈ median")

print("\n" + "-"*70)
print("VISUAL EVIDENCE:")
print("-"*70)
print(f"\n1. Histogram with Normal Curve:")
print(f"   ✓ Clear mismatch between data and expected bell curve")
print(f"   ✓ Data concentrated on left, long tail on right")

print(f"\n2. Q-Q Plot:")
print(f"   ✓ Points deviate significantly from the theoretical line")
print(f"   ✓ Curved pattern confirms right-skewness")
print(f"   ✓ Most important visual evidence of non-normality")

print(f"\n3. Density Plot:")
print(f"   ✓ Actual data density (blue) doesn't match normal curve (red)")
print(f"   ✓ Asymmetric shape with extended right tail")

print(f"\n4. Box Plot:")
print(f"   ✓ Many outliers visible (red dots above the box)")
print(f"   ✓ Median (red line) shifted toward bottom of box")
print(f"   ✓ Asymmetric box indicates skewness")

print(f"\n5. Cumulative Distribution:")
print(f"   ✓ Actual CDF (blue) diverges from theoretical CDF (red)")
print(f"   ✓ Significant deviation confirms non-normality")

print("\n" + "-"*70)
print("IMPLICATIONS FOR MODELING:")
print("-"*70)
print(f"\n✓ Since diameter is not normally distributed:")
print(f"  1. Regression models assuming normality may have issues")
print(f"  2. Consider using LOG transformation: log(diameter)")
print(f"  3. Use robust regression or non-parametric methods")
print(f"  4. Tree-based models (Random Forest, XGBoost) are NOT sensitive to normality")
print(f"  5. Deep learning models can handle non-normal distributions well")

print(f"\n✓ Log-Transformed Data:")
print(f"  • Much more normally distributed than original")
print(f"  • Skewness reduced from {stats.skew(diameter):.4f} to {stats.skew(log_diameter):.4f}")
print(f"  • Consider modeling log(diameter) instead of diameter")

print("\n" + "="*70)